In [ ]:
# Imports
from typing import Literal, List, Union, Optional, Annotated, Tuple, Callable
from collections import defaultdict
import time
import re

# Dependencies
import requests
import pandas as pd
import mwparserfromhell
from pydantic import BaseModel, Field

# Definition
JSONType = dict[str]

In [ ]:
# Classes
class	MediaWikiAPI:
	"""
	Class that contains the url for API request. It exist to make HTTP requests only.
	"""
	API_URL = "https://cardfight.fandom.com/api.php"

	def	__init__(self):
		self.session = requests.Session()

	def	get(self,
			params: dict[str, Union[str, List[str]]],
			headers: dict[str, str]) -> JSONType:
		"""
		Function to obtain information from the MediaWikiAPI. In order to use it, you
		must define ther correct HTTP parameter. The returned data will have a json structure.

		Parameters:
			params: necesary parameters to make a request to the API. Please consult https://www.mediawiki.org/wiki/API:Action_API.
			headers: HTTP headers (such as User-Agent).

		Returns:
			JSONType: If the request was succesful, you will have a json file with the desired information.
		"""
		return (
			self.session.get(	
			self.API_URL,
			params=params,
			headers=headers
			).json()
		)

class	VanguardScrapper:
	def	__init__(self, api: MediaWikiAPI):
		self.api = api

	def	obtain_main_sets(self, data: JSONType):
		sets = []
		for link in data["parse"]["links"]:
			if (link.get("ns") == 0):
				sets.append(link["*"])
		return (sets)

class	VanguardParser:
	def separate_boosters(self, data: list):
		no_main_sets = []
		for i in range(len(data) - 1, -1, -1):
			value = data[i]
			if ("Booster" not in value or "Cardfight!!" in value or "Unique" in value):
				no_main_sets.append(data.pop(i))
		return (no_main_sets)

	def remove_from_list(self, sets: list, to_delete: list):
		for i in to_delete:
			if i in sets:
				sets.remove(i)

class	VanguardClassifier:
	def	__init__(self, rules: Union[list[Tuple[str, str]]]):
		self.rules = rules

	def	obtain_set_number(self, play_set: str):
		number = ""
		for i in play_set:
			if (i.isdigit()):
				number += i
			elif (number):
				break
		return (int(number))

	def	sort_sets(self, lst: list, element: str):
		num = self.obtain_set_number(element)
		i = 0
		while (i < len(lst)):
			if (self.obtain_set_number(lst[i]) > num):
				break
			i += 1
		if (element in lst):
			return ("")
		lst.insert(i, element)

	def	classify(self, name: str) -> str:
		num = self.obtain_set_number(name)
		if (num in (16, 17)):
			return ("LL")
		for pattern, key in self.rules:
			if (re.match(pattern, name)):
				return (key)
		return ("LB")

class	VanguardStorage:
	def __init__(self):
		self.seen = {
			"LB": set(), "LL": set(), "G": set(),
			"V": set(), "D": set(), "DZ": set()
		}
		self.lb =		[]
		self.ll =		[]
		self.g =		[]
		self.v =		[]
		self.d =		[]
		self.dz	=		[]
	
		self.lb_url =	[]
		self.ll_url =	[]
		self.g_url =	[]
		self.v_url =	[]
		self.d_url =	[]
		self.dz_url	=	[]

	def _add_item(self, key: str, item: str):
		if item not in self.seen[key]:
			self.seen[key].add(item)
			getattr(self, key.lower()).append(item)

class	VanguardPipeline:
	def	__init__(self, scrapper, parser, classifier, storage):
		self.scrapper =	scrapper
		self.parser = parser
		self.classifier = classifier
		self.storage = storage

class	ScrapCard(BaseModel):
	card_nro:		str
	name:			str
	grade:			Optional[int] = None
	nation:			str
	card_type:		str | None = None
	rarity:			str
	release:		str

class	Card(BaseModel):
	name:			str
	kana:			Optional[str] = None
	phonetic:		Optional[str] = None
	thai:			Optional[str] = None
	card_type:		str
	grade:			Optional[int] = None
	skill:			Optional[str] = None
	power:			Optional[int] = None
	shield:			Optional[int] = None
	critical:		Optional[int] = None
	nation:			Optional[str] = None
	race:			Optional[str] = None
	valid_format:	Optional[str] = None
	card_set:		list[str] = Field(default_factory=list)
	card_flavor:	Optional[str] = None
	card_effect:	str
	tournament:		Optional[str] = None

class	Trigger(Card):
	card_type:		str = "trigger_unit"
	boost:			Optional[int] = None
	trigger_effect:	str

class	Order(Card):
	card_type:		str = "order"
	order_type:		str

class	NormalOrder(Order):
	order_type:		Literal["normal"]

class	BlitzOrder(Order):
	order_type:		Literal["blitz"]

CardUnion = Annotated[
	Union[Trigger, NormalOrder, BlitzOrder],
	Field(discriminator="card_type")
]

In [ ]:
# Params
urls = []
header = {
	"User-Agent": "VanguardScrapper/1.0 (Python; contact: kmarrero1993@gmail.com)"
}
params_for_urls = {
	"action": "parse",
	"page": "List_of_Cardfight!!_Vanguard_Booster_Sets",
	"format": "json"
}
rules = [
	(r"^DZ", "DZ"),
	(r"^D", "D"),
	(r"^G", "G"),
	(r"^V", "V"),
	(r"^Booster", "LB"),
]

In [ ]:
# Url Parser
web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardScrapper(web),
	VanguardParser(),
	VanguardClassifier(rules),
	VanguardStorage()
)
urls = pipeline.scrapper.api.get(params=params_for_urls, headers=header)
crude_sets = pipeline.scrapper.obtain_main_sets(urls)
no_main_sets = pipeline.parser.separate_boosters(crude_sets)
pipeline.parser.remove_from_list(no_main_sets, ["Lyrical Monasterio", "The Mask Collection"])
pipeline.parser.remove_from_list(crude_sets, [
	"G Booster Set 8: Collezione del Combattente Vol.1",
	"G Booster Set 9: Collezione del Combattente Vol.2",
	"Thailand Booster Set 1: The Mask Collection",
	"G Booster Set 7: Giudizio delle Lame Splendenti",
	"G Booster Set 1: Trascendenza Interdimensionale"
])

for i in crude_sets:
	key = pipeline.classifier.classify(i)
	pipeline.storage._add_item(key, i)

attr = ["LB", "LL", "G", "V", "D", "DZ"]
for at in attr:
	lst = getattr(pipeline.storage, at.lower())
	lst.sort(key=pipeline.classifier.obtain_set_number)
del lst


In [ ]:
x = pipeline.storage.dz.copy()

In [ ]:
pipeline.parser.make_url(x, pipeline.storage.dz_url)

In [ ]:
pipeline.storage.d_url

In [ ]:
params = {
	"action": "query",
	"format": "json",
	"prop": "revisions",
	"titles": pipeline.storage.d[1],
	"rvprop": "content",
	"rvslots": "main"
}

In [ ]:
print(params)

In [ ]:
scrap = pipeline.scrapper.api.get(params=params, headers=header)

In [ ]:
scrap

In [ ]:
for link in scrap["query"].values():
	print(link)

In [ ]:
x = [1, 2, 3, 4, 5]

In [ ]:
params_for_urls = {
	"action": "parse",
	"page": "List_of_Cardfight!!_Vanguard_Booster_Sets",
	"format": "json"
}

In [ ]:
pipeline.storage.ll